# Notebook para implementar y validar un experimento de escalabilidad controlado en QNIM

Este cuaderno documenta la ruta de trabajo para separar dimensiones PCA y qubits, corregir la ejecución en IBM y asegurar validación estadística antes de lanzar una campaña real.

## Objetivo
- Separar `n_qubits` y `pca_components` en la lógica del experimento.
- Garantizar que la dimensión del feature map coincide exactamente con la entrada del circuito.
- Garantizar que la ejecución IBM sea real (no un fake fallback).
- Añadir tests y métricas que permitan demostrar diferencia frente al azar y frente a la referencia.


## 1. Modificar ExperimentConfig para separar n_qubits y pca_components

Actualizar la clase `ExperimentConfig` para incluir `n_qubits`, `pca_components` y `n_pca_fit` por separado, con validaciones estrictas en `__post_init__`.

Requisitos:
- `n_qubits` debe estar en `[2, 32]`.
- `pca_components` debe estar en `[1, n_qubits]`.
- `n_pca_fit` debe ser `>= pca_components`.
- `train_on_hardware` solo debe permitirse cuando `mode == "ibm"`.

Esto permite distinguir entre:
- un experimento de control (16 qubits con 12 PCA), y
- un experimento de escalado (16 qubits con 16 PCA).


## 2. Actualizar LIGOPCAAdapter y generación de dataset

Modificar `LIGOPCAAdapter` para aceptar `n_components` y `n_pca_fit`, y asegurar que la generación del dataset usa una única PCA de máxima dimensión y luego selecciona prefijos para experimentos de menor dimensión.

Pautas:
- Ajustar PCA solo con `X_train_raw`.
- Transformar validación y test con la misma PCA.
- Reproducibilidad por `seed`.
- Guardar metadatos de varianza explicada y componentes seleccionadas.


## 3. Actualizar el pipeline de PCA y validación de dimensiones

Asegurar que el pipeline de PCA:
- ajusta sobre el conjunto de entrenamiento,
- transforma validación y test sin re-ajuste,
- valida que las dimensiones de entrada coinciden exactamente con las requeridas por el circuito.

No debe haber relleno automático con ceros; un dataset con 12 features no debe ejecutarse con 32 qubits sin una validación explícita.


## 4. Actualizar construcción de circuitos en QiskitVQCTrainer

Modificar `_build_feature_map_and_ansatz` para aceptar `n_qubits` y `n_features` de forma independiente, teniendo en cuenta que:
- el `ZZFeatureMap` usa las componentes PCA disponibles,
- el `ansatz` opera sobre todos los qubits del circuito,
- la codificación del input no debe confundirse con la capacidad del circuito.

Esto permite comparar resultados entre:
- 16 qubits / 12 PCA,
- 16 qubits / 16 PCA,
- 32 qubits / 12 PCA,
- 32 qubits / 32 PCA.


## 5. Actualizar funciones de preparación de features y propagación de dimensiones

Reemplazar el relleno automático con validaciones explícitas en:
- `_prepare_feature_matrix()`
- `_prepare_feature_vector()`
- `_make_vqc_loss_fn()`
- `train_vqc()`
- `train_and_evaluate()`
- `_evaluate_real_predictions()`
- `_validate_on_ibm()`

La regla es simple: si el circuito espera `n_features`, el dataset debe tener exactamente ese número de columnas. Si no, se debe lanzar error y no se debe permitir el experimento.


## 6. Corregir entrenamiento y validación en IBM Quantum

Asegurar que la ruta IBM sea real:
- pasar `IBM_QUANTUM_TOKEN` al entrenador,
- seleccionar sampler y backend correctos,
- propagar los errores reales si la validación en hardware falla,
- eliminar cualquier fallback que devuelva accuracies inventadas.

La regla de oro es: un resultado IBM solo es válido si existe un job real y una medición real recuperada del backend.


## 7. Implementar métricas estadísticas y validación experimental

Crear `src/application/scaling_metrics.py` con funciones de:
- `stratified_bootstrap()`
- `permutation_test()`
- `paired_bootstrap_difference()`

Estas funciones permiten calcular IC95, p-values ajustados y diferencias emparejadas entre configuraciones usando el mismo conjunto de eventos.

Objetivo: demostrar si una configuración supera el azar y si supera a la referencia de 12 qubits con una diferencia estadísticamente significativa.


## 8. Actualizar generación de datos y partición train/val/test

Modificar el dataset para incluir conjuntos independientes de entrenamiento, validación y test.

Criterios:
- Separar los ejemplos antes de ajustar PCA y cualquier scaler.
- Ajustar PCA/scaler exclusivamente con entrenamiento.
- Transformar validación y test con los objetos ajustados en train.
- No utilizar test para selección de hiperparámetros ni fine-tuning.
- Mantener particiones estratificadas y reproducibles.

El conjunto final de test debe ser compartido por todas las configuraciones que se comparen.


## 9. Crear ejecutor automático de experimentos de escalabilidad

Implementar `run_scaling_study_use_case.py` y `scripts/run_scaling_study.py` para ejecutar la matriz de experimentos de forma reproducible.

La matriz recomendada incluye:

| ID | Qubits | PCA | Objetivo |
|---|---:|---:|---|
| E1 | 12 | 12 | Referencia |
| E2 | 16 | 12 | Control de qubits |
| E3 | 16 | 16 | Escalado conjunto |
| E4 | 20 | 12 | Control de qubits |
| E5 | 20 | 20 | Escalado conjunto |
| E6 | 32 | 12 | Control de qubits |
| E7 | 32 | 32 | Escalado conjunto |

Guardar por ejecución la configuración, semilla, hash del dataset/commit, métricas, predicciones y metadatos de hardware cuando existan. Los siete experimentos deben usar exactamente el mismo test.


## 10. Actualizar perfiles experimentales y CLI

Añadir perfiles para las configuraciones 12/12, 16/16, 20/20 y 32/32, además de controles de 16/12, 20/12 y 32/12.

Actualizar `scripts/generate_results.py` para aceptar y propagar:
- `--n-qubits`
- `--pca-components`
- `--n-pca-fit`
- `--train-on-hardware`
- `--hardware-shots`
- `--backend`

En modo `--from-disk`, validar que el dataset dispone de suficientes columnas y seleccionar el prefijo PCA requerido. Aplicar lo mismo a train, validation y test.


## 11. Añadir auditoría de información PCA

Calcular y registrar dos diagnósticos por componente:
- varianza explicada y varianza acumulada,
- Fisher score y p-value de asociación con las clases, calculados solo sobre entrenamiento.

Estos valores sirven para describir la información retenida y su poder discriminativo. No se deben usar las etiquetas del conjunto de test para seleccionar componentes ni ajustar el modelo.


## 12. Implementar y ejecutar tests unitarios y de integración

Añadir tests para:
- Dimensiones válidas: 12/12, 16/16, 32/32 y 32/12.
- Rechazo de 12 qubits / 16 PCA y de cualquier discrepancia entre features y circuito.
- Prefijos PCA reproducibles a partir de una única PCA de dimensión máxima.
- Métricas bootstrap, permutation test y comparación emparejada.
- Rutas de ejecución IBM mediante mocks/samplers simulados, sin consumir créditos.
- Propagación de errores IBM y ausencia de métricas ficticias.
- Partición estratificada e independencia del conjunto test.

## Criterio de aceptación experimental

Una configuración supera el azar si el límite inferior unilateral del IC de balanced accuracy está por encima de `1/13`, y el permutation test corregido por Holm tiene `p < 0.05`. Para afirmar mejora frente a 12 qubits, el límite inferior del IC de la diferencia emparejada también debe ser mayor que cero. Registrar múltiples semillas para evaluar reproducibilidad.

## Orden de ejecución recomendado

1. Preparar un único dataset de dimensión PCA máxima y sus particiones independientes.
2. Verificar PCA, dimensiones y circuitos con tests.
3. Ejecutar barrido en simulador ideal.
4. Repetir con ruido calibrado del backend.
5. Seleccionar configuraciones candidatas según criterios predefinidos.
6. Hacer fine-tuning en hardware con presupuesto acotado.
7. Evaluar todas las configuraciones en el mismo test independiente y guardar métricas/predicciones reales.

No se debe afirmar una mejora antes de tener resultados medidos. QGT del optimizador tampoco debe presentarse como una medición de QFI física.
